<a href="https://colab.research.google.com/github/MaynineHuang/COMPSCI764_RAG/blob/main/notebooks/Meihan_predefine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/MaynineHuang/COMPSCI764_RAG.git
%cd COMPSCI764_RAG

Cloning into 'COMPSCI764_RAG'...
remote: Enumerating objects: 90, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 90 (delta 42), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (90/90), 88.38 KiB | 1.67 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/COMPSCI764_RAG


In [2]:
!pip install -q datasets sentence-transformers

In [3]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

In [4]:
#load the data
hotpot = load_dataset(
        "hotpotqa/hotpot_qa",
        "distractor",
        split="validation[:5]"
         )
print(hotpot)

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 5
})


In [5]:
#load the embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
retrieval_model = SentenceTransformer(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2


In [6]:
example = hotpot[0]
print("Question:")
print(example["question"])
print("\nGold answer:")
print(example["answer"])
print("\nGold supporting facts:")
print(example["supporting_facts"])

Question:
Were Scott Derrickson and Ed Wood of the same nationality?

Gold answer:
yes

Gold supporting facts:
{'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}


In [7]:
#从这一步, get the question
titles = example["context"]["title"]
sentence_lists = example["context"]["sentences"]
passages = []
for title, sentences in zip(titles, sentence_lists):
     passage = title + ". " + " ".join(sentences)
     passages.append(passage)
print("Number of passages:", len(passages))

Number of passages: 10


In [8]:
#get the passages
question_embedding = retrieval_model.encode(
    example["question"],
    convert_to_tensor=True
)
passage_embeddings = retrieval_model.encode(
    passages,
    convert_to_tensor=True
)
print("Question embedding shape:", question_embedding.shape)
print("Passage embeddings shape:", passage_embeddings.shape)

Question embedding shape: torch.Size([384])
Passage embeddings shape: torch.Size([10, 384])


In [9]:
#Calculate cosine similarity
scores = util.cos_sim(
    question_embedding,
    passage_embeddings
)[0]
print(scores)

tensor([0.4753, 0.2636, 0.2922, 0.1315, 0.5520, 0.0636, 0.3287, 0.0818, 0.3027,
        0.0329])


In [10]:
#rank all passages
ranked_indices = scores.argsort(descending=True)
for rank, idx in enumerate(ranked_indices):
   idx = int(idx)
   print("\nRank:", rank + 1)
   print("Score:", round(float(scores[idx]), 4))
   print("Title:", titles[idx])


Rank: 1
Score: 0.552
Title: Ed Wood

Rank: 2
Score: 0.4753
Title: Ed Wood (film)

Rank: 3
Score: 0.3287
Title: Adam Collis

Rank: 4
Score: 0.3027
Title: Conrad Brooks

Rank: 5
Score: 0.2922
Title: Woodson, Arkansas

Rank: 6
Score: 0.2636
Title: Scott Derrickson

Rank: 7
Score: 0.1315
Title: Tyler Bates

Rank: 8
Score: 0.0818
Title: Sinister (film)

Rank: 9
Score: 0.0636
Title: Deliver Us from Evil (2014 film)

Rank: 10
Score: 0.0329
Title: Doctor Strange (2016 film)


In [11]:
#keep only the top 8 passages
TOP_K = 8
top_indices = ranked_indices[:TOP_K]
retrieved_titles = []
print("TOP 5 RETRIEVED PASSAGES")
for rank, idx in enumerate(top_indices):
  idx = int(idx)

  retrieved_titles.append(titles[idx])

  print("\nRank:", rank + 1)
  print("Score:", round(float(scores[idx]), 4))
  print("Title:", titles[idx])

TOP 5 RETRIEVED PASSAGES

Rank: 1
Score: 0.552
Title: Ed Wood

Rank: 2
Score: 0.4753
Title: Ed Wood (film)

Rank: 3
Score: 0.3287
Title: Adam Collis

Rank: 4
Score: 0.3027
Title: Conrad Brooks

Rank: 5
Score: 0.2922
Title: Woodson, Arkansas

Rank: 6
Score: 0.2636
Title: Scott Derrickson

Rank: 7
Score: 0.1315
Title: Tyler Bates

Rank: 8
Score: 0.0818
Title: Sinister (film)


In [12]:
#measure supporting-passage evidene recall
gold_support_titles = set(example["supporting_facts"]["title"])
retrieved_title_set = set(retrieved_titles)

found_titles = gold_support_titles.intersection(retrieved_title_set)

evidence_recall = len(found_titles) / len(gold_support_titles)

print("Gold supporting titles:", gold_support_titles)
print("Retrieved titles:", retrieved_title_set)
print("Found supporting titles:", found_titles)
print("Evidence recall:", evidence_recall)

Gold supporting titles: {'Ed Wood', 'Scott Derrickson'}
Retrieved titles: {'Conrad Brooks', 'Scott Derrickson', 'Ed Wood', 'Woodson, Arkansas', 'Ed Wood (film)', 'Adam Collis', 'Sinister (film)', 'Tyler Bates'}
Found supporting titles: {'Ed Wood', 'Scott Derrickson'}
Evidence recall: 1.0


In [13]:
#define the retrieve_top_k
def retrieve_top_k(example, model, top_k=8):
  question = example["question"]
  titles = example["context"]["title"]
  sentence_lists = example["context"]["sentences"]

  passages = [
      title + ". " + " ".join(sentences)
      for title, sentences in zip(titles, sentence_lists)
  ]
  question_embedding = model.encode(
      question,
      convert_to_tensor=True
  )

  passage_embeddings = model.encode(
      passages,
      convert_to_tensor=True
  )

  scores = util.cos_sim(
      question_embedding,
      passage_embeddings    )[0]

  ranked_indices = scores.argsort(descending=True)
  top_indices = ranked_indices[:top_k]

  results = []

  for rank, idx in enumerate(top_indices):
    idx = int(idx)
    results.append({
        "rank": rank + 1,
        "title": titles[idx],
        "passage": passages[idx],
        "score": float(scores[idx])
        })

  return results

In [14]:
#test the function on example1
results = retrieve_top_k(
    hotpot[0],
    retrieval_model,
    top_k=8
)
for result in results:
  print(
      result["rank"],
      round(result["score"], 4),
      result["title"]
  )

1 0.552 Ed Wood
2 0.4753 Ed Wood (film)
3 0.3287 Adam Collis
4 0.3027 Conrad Brooks
5 0.2922 Woodson, Arkansas
6 0.2636 Scott Derrickson
7 0.1315 Tyler Bates
8 0.0818 Sinister (film)


In [15]:
for i in range(len(hotpot)):
  example = hotpot[i]

  results = retrieve_top_k(
      example,
      retrieval_model,
      top_k=8
  )

  retrieved_titles = {
      result["title"]
      for result in results
  }

  gold_titles = set(
      example["supporting_facts"]["title"]
  )

  found = gold_titles.intersection(retrieved_titles)

  recall = len(found) / len(gold_titles)

  print("\n" + "=" * 70)
  print("EXAMPLE", i + 1)
  print("Question:", example["question"])
  print("Gold answer:", example["answer"])
  print("Gold supporting titles:", gold_titles)

  print("\nTop 8:")
  for result in results:
    print(
        result["rank"],
        "|",
        round(result["score"], 4),
        "|",
        result["title"]
    )

  print("\nEvidence recall:", recall)


EXAMPLE 1
Question: Were Scott Derrickson and Ed Wood of the same nationality?
Gold answer: yes
Gold supporting titles: {'Ed Wood', 'Scott Derrickson'}

Top 8:
1 | 0.552 | Ed Wood
2 | 0.4753 | Ed Wood (film)
3 | 0.3287 | Adam Collis
4 | 0.3027 | Conrad Brooks
5 | 0.2922 | Woodson, Arkansas
6 | 0.2636 | Scott Derrickson
7 | 0.1315 | Tyler Bates
8 | 0.0818 | Sinister (film)

Evidence recall: 1.0

EXAMPLE 2
Question: What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?
Gold answer: Chief of Protocol
Gold supporting titles: {'Kiss and Tell (1945 film)', 'Shirley Temple'}

Top 8:
1 | 0.6683 | Kiss and Tell (1945 film)
2 | 0.5741 | A Kiss for Corliss
3 | 0.5452 | Meet Corliss Archer (TV series)
4 | 0.3984 | Janet Waldo
5 | 0.3342 | Meet Corliss Archer
6 | 0.305 | Shirley Temple
7 | 0.2271 | Charles Craft
8 | 0.1181 | Secretary of State for Constitutional Affairs

Evidence recall: 1.0

EXAMPLE 3
Question: What science fantasy young adult seri

In [16]:
# Final Retrieval V1 validation on 20 examples

hotpot_20 = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split="validation[:20]"
)

recalls = []

for i in range(20):
    example = hotpot_20[i]

    results = retrieve_top_k(
        example,
        retrieval_model,
        top_k=8
    )

    retrieved_titles = {
        result["title"]
        for result in results
    }

    gold_titles = set(
        example["supporting_facts"]["title"]
    )

    found_titles = gold_titles.intersection(retrieved_titles)

    recall = len(found_titles) / len(gold_titles)
    recalls.append(recall)

    print(
        f"Example {i + 1}: "
        f"Recall@5 = {recall:.2f} | "
        f"Found {len(found_titles)}/{len(gold_titles)} supporting passages"
    )

average_recall = sum(recalls) / len(recalls)

full_recall_count = sum(
    recall == 1.0
    for recall in recalls
)

print("\n" + "=" * 60)
print("RETRIEVAL V1 - 20 EXAMPLE SUMMARY")
print("=" * 60)
print("Average Evidence Recall@5:", round(average_recall, 3))
print("Questions with full supporting evidence:", full_recall_count, "/ 20")
print(
    "Full evidence retrieval rate:",
    round(full_recall_count / 20, 3)
)

Example 1: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 2: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 3: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 4: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 5: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 6: Recall@5 = 0.50 | Found 1/2 supporting passages
Example 7: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 8: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 9: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 10: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 11: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 12: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 13: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 14: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 15: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 16: Recall@5 = 1.00 | Found 2/2 supporting passages
Example 17: Recall@5 = 0.50 | Found 1/2 supportin

In [17]:
# Retrieval V1 - Final validation on 20 HotpotQA examples

!pip install -q datasets sentence-transformers

from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util


# Settings
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 8
NUM_EXAMPLES = 20


# Load 20 validation examples
hotpot_20 = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split=f"validation[:{NUM_EXAMPLES}]"
)

print("Loaded examples:", len(hotpot_20))


# Load retrieval model
retrieval_model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)


# Retrieval function
def retrieve_top_k(example, model, top_k=5):
    question = example["question"]
    titles = example["context"]["title"]
    sentence_lists = example["context"]["sentences"]

    passages = [
        title + ". " + " ".join(sentences)
        for title, sentences in zip(titles, sentence_lists)
    ]

    question_embedding = model.encode(
        question,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    passage_embeddings = model.encode(
        passages,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(
        question_embedding,
        passage_embeddings
    )[0]

    top_k = min(top_k, len(passages))

    ranked_indices = scores.argsort(descending=True)
    top_indices = ranked_indices[:top_k]

    results = []

    for rank, idx in enumerate(top_indices):
        idx = int(idx)

        results.append({
            "rank": rank + 1,
            "title": titles[idx],
            "passage": passages[idx],
            "score": float(scores[idx])
        })

    return results


# Evaluate Retrieval V1
recalls = []
full_recall_count = 0

for i in range(NUM_EXAMPLES):
    example = hotpot_20[i]

    results = retrieve_top_k(
        example,
        retrieval_model,
        top_k=TOP_K
    )

    retrieved_titles = {
        result["title"]
        for result in results
    }

    gold_titles = set(
        example["supporting_facts"]["title"]
    )

    found_titles = gold_titles.intersection(retrieved_titles)

    recall = len(found_titles) / len(gold_titles)
    recalls.append(recall)

    if recall == 1.0:
        full_recall_count += 1

    print(
        f"Example {i + 1:02d} | "
        f"Recall@{TOP_K}: {recall:.2f} | "
        f"Found: {len(found_titles)}/{len(gold_titles)}"
    )


# Final summary
average_recall = sum(recalls) / len(recalls)
full_retrieval_rate = full_recall_count / NUM_EXAMPLES

print("\n" + "=" * 60)
print("RETRIEVAL V1 - 20 EXAMPLE SUMMARY")
print("=" * 60)

print(
    f"Average Evidence Recall@{TOP_K}: "
    f"{average_recall:.3f}"
)

print(
    "Questions with full supporting evidence:",
    f"{full_recall_count}/{NUM_EXAMPLES}"
)

print(
    "Full evidence retrieval rate:",
    f"{full_retrieval_rate:.3f}"
)

Loaded examples: 20


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2
Example 01 | Recall@8: 1.00 | Found: 2/2
Example 02 | Recall@8: 1.00 | Found: 2/2
Example 03 | Recall@8: 1.00 | Found: 2/2
Example 04 | Recall@8: 1.00 | Found: 2/2
Example 05 | Recall@8: 1.00 | Found: 2/2
Example 06 | Recall@8: 0.50 | Found: 1/2
Example 07 | Recall@8: 1.00 | Found: 2/2
Example 08 | Recall@8: 1.00 | Found: 2/2
Example 09 | Recall@8: 1.00 | Found: 2/2
Example 10 | Recall@8: 1.00 | Found: 2/2
Example 11 | Recall@8: 1.00 | Found: 2/2
Example 12 | Recall@8: 1.00 | Found: 2/2
Example 13 | Recall@8: 1.00 | Found: 2/2
Example 14 | Recall@8: 1.00 | Found: 2/2
Example 15 | Recall@8: 1.00 | Found: 2/2
Example 16 | Recall@8: 1.00 | Found: 2/2
Example 17 | Recall@8: 0.50 | Found: 1/2
Example 18 | Recall@8: 1.00 | Found: 2/2
Example 19 | Recall@8: 0.50 | Found: 1/2
Example 20 | Recall@8: 1.00 | Found: 2/2

RETRIEVAL V1 - 20 EXAMPLE SUMMARY
Average Evidence Recall@8: 0.925
Questions with full supporting evidence: 17/20
Full evidenc

In [18]:
# Compare different Top-k values before freezing retrieval

TOP_K_VALUES = [5, 6, 7, 8]

for top_k in TOP_K_VALUES:
    recalls = []
    full_recall_count = 0

    for i in range(len(hotpot_20)):
        example = hotpot_20[i]

        results = retrieve_top_k(
            example,
            retrieval_model,
            top_k=top_k
        )

        retrieved_titles = {
            result["title"]
            for result in results
        }

        gold_titles = set(
            example["supporting_facts"]["title"]
        )

        found_titles = gold_titles.intersection(retrieved_titles)

        recall = len(found_titles) / len(gold_titles)
        recalls.append(recall)

        if recall == 1.0:
            full_recall_count += 1

    average_recall = sum(recalls) / len(recalls)
    full_retrieval_rate = full_recall_count / len(hotpot_20)

    print(
        f"Top-{top_k} | "
        f"Average Recall: {average_recall:.3f} | "
        f"Full Evidence: {full_recall_count}/20 | "
        f"Full Evidence Rate: {full_retrieval_rate:.3f}"
    )

Top-5 | Average Recall: 0.775 | Full Evidence: 11/20 | Full Evidence Rate: 0.550
Top-6 | Average Recall: 0.825 | Full Evidence: 13/20 | Full Evidence Rate: 0.650
Top-7 | Average Recall: 0.875 | Full Evidence: 15/20 | Full Evidence Rate: 0.750
Top-8 | Average Recall: 0.925 | Full Evidence: 17/20 | Full Evidence Rate: 0.850


## Step 8+9

In [19]:
# Oct 15 2026 new-update meihan
!pip install -q transformers sentencepiece accelerate


In [20]:

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GENERATOR_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_NAME)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATOR_NAME
)

device = "cuda" if torch.cuda.is_available() else "cpu"
generator_model = generator_model.to(device)
generator_model.eval()

print("Generator loaded:", GENERATOR_NAME)
print("Device:", device)


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator loaded: google/flan-t5-base
Device: cpu


In [21]:

TOP_K = 8

def generate_rag_answer(question, retrieved_results):
    """
    Generate an answer using the question and retrieved passages.
    No attack injection and no defence are applied.
    """

    context = "\n\n".join(
        [
            f"Passage {item['rank']} "
            f"(Title: {item['title']}):\n{item['passage']}"
            for item in retrieved_results
        ]
    )

    prompt = f"""Answer the question using the context below.
Give a short and direct answer.
Do not add explanations unless necessary.
If the answer cannot be found in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        outputs = generator_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer.strip()


In [22]:

# Select one example from the validation dataset
example = hotpot_20[0]

# Retrieve the top 8 passages
retrieved_results = retrieve_top_k(
    example,
    retrieval_model,
    top_k=TOP_K
)

# Generate the answer
predicted_answer = generate_rag_answer(
    example["question"],
    retrieved_results
)

print("Question:")
print(example["question"])

print("\nGold answer:")
print(example["answer"])

print("\nPredicted answer:")
print(predicted_answer)

print("\nRetrieved passages:")
for item in retrieved_results:
    print(
        item["rank"],
        "|",
        round(item["score"], 4),
        "|",
        item["title"]
    )


Question:
Were Scott Derrickson and Ed Wood of the same nationality?

Gold answer:
yes

Predicted answer:
Yes.

Retrieved passages:
1 | 0.552 | Ed Wood
2 | 0.4753 | Ed Wood (film)
3 | 0.3287 | Adam Collis
4 | 0.3027 | Conrad Brooks
5 | 0.2922 | Woodson, Arkansas
6 | 0.2636 | Scott Derrickson
7 | 0.1315 | Tyler Bates
8 | 0.0818 | Sinister (film)


In [23]:

import re
import string
from collections import Counter

def normalize_answer(text):
    """Normalize an answer for HotpotQA-style evaluation."""
    text = text.lower()
    text = "".join(
        char for char in text
        if char not in string.punctuation
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = " ".join(text.split())
    return text


def exact_match_score(prediction, reference):
    return int(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def token_f1_score(prediction, reference):
    pred_tokens = normalize_answer(prediction).split()
    ref_tokens = normalize_answer(reference).split()

    if not pred_tokens or not ref_tokens:
        return float(pred_tokens == ref_tokens)

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)


In [24]:

baseline_results = []

for i, example in enumerate(hotpot_20):
    question = example["question"]
    gold_answer = example["answer"]

    # Step 1: Retrieve top-8 passages
    retrieved = retrieve_top_k(
        example,
        retrieval_model,
        top_k=TOP_K
    )

    # Step 2: Generate an answer without attack or defence
    prediction = generate_rag_answer(
        question,
        retrieved
    )

    # Step 3: Evaluate answer quality
    em = exact_match_score(prediction, gold_answer)
    f1 = token_f1_score(prediction, gold_answer)

    # Step 4: Evaluate supporting-passage recall
    retrieved_titles = {
        item["title"] for item in retrieved
    }
    gold_titles = set(example["supporting_facts"]["title"])

    if gold_titles:
        evidence_recall = (
            len(gold_titles & retrieved_titles)
            / len(gold_titles)
        )
    else:
        evidence_recall = 0.0

    baseline_results.append({
        "id": example["id"],
        "question": question,
        "gold_answer": gold_answer,
        "prediction": prediction,
        "EM": em,
        "Token_F1": f1,
        "Evidence_Recall": evidence_recall
    })

    print(
        f"Example {i + 1}/{len(hotpot_20)} | "
        f"EM={em} | "
        f"F1={f1:.3f} | "
        f"Evidence Recall={evidence_recall:.3f}"
    )

print("\n===== CLEAN RAG BASELINE =====")

print(
    "Average EM:",
    round(
        sum(r["EM"] for r in baseline_results)
        / len(baseline_results),
        3
    )
)

print(
    "Average Token F1:",
    round(
        sum(r["Token_F1"] for r in baseline_results)
        / len(baseline_results),
        3
    )
)

print(
    "Average Evidence Recall:",
    round(
        sum(r["Evidence_Recall"] for r in baseline_results)
        / len(baseline_results),
        3
    )
)


Example 1/20 | EM=1 | F1=1.000 | Evidence Recall=1.000
Example 2/20 | EM=0 | F1=0.222 | Evidence Recall=1.000
Example 3/20 | EM=1 | F1=1.000 | Evidence Recall=1.000
Example 4/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 5/20 | EM=0 | F1=0.571 | Evidence Recall=1.000
Example 6/20 | EM=0 | F1=0.000 | Evidence Recall=0.500
Example 7/20 | EM=1 | F1=1.000 | Evidence Recall=1.000
Example 8/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 9/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 10/20 | EM=1 | F1=1.000 | Evidence Recall=1.000
Example 11/20 | EM=0 | F1=0.400 | Evidence Recall=1.000
Example 12/20 | EM=1 | F1=1.000 | Evidence Recall=1.000
Example 13/20 | EM=1 | F1=1.000 | Evidence Recall=1.000
Example 14/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 15/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 16/20 | EM=0 | F1=0.000 | Evidence Recall=1.000
Example 17/20 | EM=0 | F1=0.000 | Evidence Recall=0.500
Example 18/20 | EM=1 | F1=1.000 | Evidence Recall=1.000
E

In [25]:

import pandas as pd

# Save the existing baseline results
baseline_df = pd.DataFrame(baseline_results)

baseline_df.to_csv(
    "clean_rag_baseline.csv",
    index=False
)

print("Number of examples:", len(baseline_df))
print("\nAverage metrics:")
print(
    baseline_df[["EM", "Token_F1", "Evidence_Recall"]]
    .mean()
)

print("\nMissing values:")
print(
    baseline_df[
        ["id", "prediction", "EM", "Token_F1", "Evidence_Recall"]
    ].isna().sum()
)

print("\nSaved: clean_rag_baseline.csv")


Number of examples: 20

Average metrics:
EM                 0.350000
Token_F1           0.439683
Evidence_Recall    0.925000
dtype: float64

Missing values:
id                 0
prediction         0
EM                 0
Token_F1           0
Evidence_Recall    0
dtype: int64

Saved: clean_rag_baseline.csv


In [26]:

import pandas as pd

df = pd.DataFrame(baseline_results)

# Inspect questions that were not answered exactly correctly
errors = df[df["EM"] == 0][
    [
        "id",
        "question",
        "gold_answer",
        "prediction",
        "EM",
        "Token_F1",
        "Evidence_Recall"
    ]
]

print("Number of EM errors:", len(errors))
display(errors)


Number of EM errors: 13


,id,question,gold_answer,prediction,EM,Token_F1,Evidence_Recall
1,5a8c7595554299585d9e36b6,What government position was held by the woman...,Chief of Protocol,Secretary of State for Constitutional Affairs.,0,0.222222,1.0
3,5adbf0a255429947ff17385a,Are the Laleli Mosque and Esma Sultan Mansion ...,no,Yes.,0,0.000000,1.0
4,5a8e3ea95542995a26add48d,"The director of the romantic comedy ""Big Stone...","Greenwich Village, New York City",Greenwich Village.,0,0.571429,1.0
5,5abd94525542992ac4f382d2,2014 S/S is the debut album of a South Korean ...,YG Entertainment,WINNER.,0,0.000000,0.5
7,5a87ab905542996e4f3088c1,The arena where the Lewiston Maineiacs played ...,"3,677 seated","4,000",0,0.000000,1.0
8,5a7bbb64554299042af8f7cc,"Who is older, Annie Morton or Terry Richardson?",Terry Richardson,Annie Morton.,0,0.000000,1.0
10,5a7166395542994082a3e814,What is the name of the fight song of the univ...,Kansas Song,We’re From Kansas.,0,0.400000,1.0
13,5ab56e32554299637185c594,Are Random House Tower and 888 7th Avenue both...,no,The key of the crash was not the escrow.,0,0.000000,1.0
14,5ab6d09255429954757d337d,The football manager who recruited David Beckh...,from 1986 to 2013,Between 1945 and 1969.,0,0.000000,1.0
15,5a75e05c55429976ec32bc5f,Brown State Fishing Lake is in a country that ...,"9,984",Brown State Fishing Lake is in the United States.,0,0.000000,1.0


In [27]:

# Check input token length for the first example
example = hotpot_20[0]

retrieved = retrieve_top_k(
    example,
    retrieval_model,
    top_k=8
)

context = "\n\n".join(
    [
        f"Passage {item['rank']} "
        f"(Title: {item['title']}):\n{item['passage']}"
        for item in retrieved
    ]
)

prompt = f"""Answer the question using the context below.
Give a short and direct answer.
Do not add explanations unless necessary.
If the answer cannot be found in the context, say you do not know.

Context:
{context}

Question: {example["question"]}

Answer:"""

encoded = tokenizer(
    prompt,
    truncation=False
)

print("Input token count:", len(encoded["input_ids"]))
print("Configured input limit:", 2048)
print("Exceeds limit:", len(encoded["input_ids"]) > 2048)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1035 > 512). Running this sequence through the model will result in indexing errors


Input token count: 1035
Configured input limit: 2048
Exceeds limit: False


not really accurate but work smoothly

## Step 13

In [28]:
import pandas as pd
from pathlib import Path

PILOT_DIR = Path("results/pilot")
PILOT_DIR.mkdir(parents=True, exist_ok=True)

# Use the same 20 validation examples already tested
pilot_questions = pd.DataFrame([
    {
        "question_id": example["id"],
        "question": example["question"],
        "gold_answer": example["answer"]
    }
    for example in hotpot_20
])

pilot_questions.to_csv(
    PILOT_DIR / "pilot_questions.csv",
    index=False
)

print("Pilot questions:", len(pilot_questions))
print(pilot_questions.head())

Pilot questions: 20
                question_id  \
0  5a8b57f25542995d1e6f1371   
1  5a8c7595554299585d9e36b6   
2  5a85ea095542994775f606a8   
3  5adbf0a255429947ff17385a   
4  5a8e3ea95542995a26add48d   

                                            question  \
0  Were Scott Derrickson and Ed Wood of the same ...   
1  What government position was held by the woman...   
2  What science fantasy young adult series, told ...   
3  Are the Laleli Mosque and Esma Sultan Mansion ...   
4  The director of the romantic comedy "Big Stone...   

                        gold_answer  
0                               yes  
1                 Chief of Protocol  
2                         Animorphs  
3                                no  
4  Greenwich Village, New York City  


In [29]:
import json

retrieval_cache = []

for example in hotpot_20:
    retrieved = retrieve_top_k(
        example,
        retrieval_model,
        top_k=8
    )

    retrieval_cache.append({
        "question_id": example["id"],
        "question": example["question"],
        "gold_answer": example["answer"],
        "retrieved_passages": [
            {
                "rank": item["rank"],
                "title": item["title"],
                "passage": item["passage"],
                "score": float(item["score"])
            }
            for item in retrieved
        ]
    })

with open(
    PILOT_DIR / "retrieval_cache.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        retrieval_cache,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Cached questions:", len(retrieval_cache))
print("Passages per question:", len(retrieval_cache[0]["retrieved_passages"]))
print("Saved to:", PILOT_DIR / "retrieval_cache.json")

Cached questions: 20
Passages per question: 8
Saved to: results/pilot/retrieval_cache.json


In [30]:

import json
import time
import pandas as pd
from pathlib import Path

# ---------- 1. Load the cached pilot data ----------
PILOT_DIR = Path("results/pilot")
PILOT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_PATH = PILOT_DIR / "retrieval_cache.json"
OUTPUT_PATH = PILOT_DIR / "clean_pilot_runner_v1.csv"

with open(CACHE_PATH, "r", encoding="utf-8") as f:
    retrieval_cache = json.load(f)

print("Loaded questions:", len(retrieval_cache))
print("Output:", OUTPUT_PATH)


# ---------- 2. Run the clean condition ----------
pilot_results = []

for i, item in enumerate(retrieval_cache):
    question_id = item["question_id"]
    question = item["question"]
    gold_answer = item["gold_answer"]
    retrieved = item["retrieved_passages"]

    start_time = time.perf_counter()

    try:
        # Reuse cached Top-8 passages; do not retrieve again
        prediction = generate_rag_answer(
            question,
            retrieved
        )

        elapsed = time.perf_counter() - start_time

        em = exact_match_score(prediction, gold_answer)
        f1 = token_f1_score(prediction, gold_answer)

        pilot_results.append({
            "question_id": question_id,
            "condition": "clean",
            "method": "baseline",
            "gold_answer": gold_answer,
            "predicted_answer": prediction,
            "EM": em,
            "Token_F1": f1,
            "num_retrieved_passages": len(retrieved),
            "generation_latency_seconds": elapsed,
            "status": "success",
            "error": None
        })

    except Exception as e:
        elapsed = time.perf_counter() - start_time

        pilot_results.append({
            "question_id": question_id,
            "condition": "clean",
            "method": "baseline",
            "gold_answer": gold_answer,
            "predicted_answer": None,
            "EM": None,
            "Token_F1": None,
            "num_retrieved_passages": len(retrieved),
            "generation_latency_seconds": elapsed,
            "status": "error",
            "error": repr(e)
        })

    print(
        f"{i + 1}/{len(retrieval_cache)} | "
        f"ID={question_id} | "
        f"{pilot_results[-1]['status']}"
    )


# ---------- 3. Save results to a NEW file ----------
pilot_df = pd.DataFrame(pilot_results)

pilot_df.to_csv(OUTPUT_PATH, index=False)

successful = pilot_df[pilot_df["status"] == "success"]

print("\n===== PILOT RUN SUMMARY =====")
print("Total:", len(pilot_df))
print("Successful:", len(successful))
print("Failed:", len(pilot_df) - len(successful))

if len(successful) > 0:
    print("Mean EM:", round(successful["EM"].mean(), 3))
    print("Mean Token F1:", round(successful["Token_F1"].mean(), 3))
    print(
        "Mean generation latency (seconds):",
        round(successful["generation_latency_seconds"].mean(), 3)
    )

print("\nSaved:", OUTPUT_PATH)


Loaded questions: 20
Output: results/pilot/clean_pilot_runner_v1.csv
1/20 | ID=5a8b57f25542995d1e6f1371 | success
2/20 | ID=5a8c7595554299585d9e36b6 | success
3/20 | ID=5a85ea095542994775f606a8 | success
4/20 | ID=5adbf0a255429947ff17385a | success
5/20 | ID=5a8e3ea95542995a26add48d | success
6/20 | ID=5abd94525542992ac4f382d2 | success
7/20 | ID=5a85b2d95542997b5ce40028 | success
8/20 | ID=5a87ab905542996e4f3088c1 | success
9/20 | ID=5a7bbb64554299042af8f7cc | success
10/20 | ID=5a8db19d5542994ba4e3dd00 | success
11/20 | ID=5a7166395542994082a3e814 | success
12/20 | ID=5a877e5d5542993e715abf7d | success
13/20 | ID=5ab3b0bf5542992ade7c6e39 | success
14/20 | ID=5ab56e32554299637185c594 | success
15/20 | ID=5ab6d09255429954757d337d | success
16/20 | ID=5a75e05c55429976ec32bc5f | success
17/20 | ID=5ab3e45655429976abd1bcd4 | success
18/20 | ID=5ab29c24554299449642c932 | success
19/20 | ID=5ae0d4c9554299603e418468 | success
20/20 | ID=5a8133725542995ce29dcbdb | success

===== PILOT RUN SUM

In [31]:
#for testing this part
import json
from pathlib import Path
import pandas as pd

# 1. Check existing pilot files
cache_path = Path("results/pilot/retrieval_cache.json")
questions_path = Path("results/pilot/pilot_questions.csv")
runner_path = Path("results/pilot/clean_pilot_runner_v1.csv")

for path in [cache_path, questions_path, runner_path]:
    print(f"{path}: {'FOUND' if path.exists() else 'MISSING'}")

# 2. Validate retrieval cache
with open(cache_path, "r", encoding="utf-8") as f:
    retrieval_cache = json.load(f)

print("\n--- Retrieval cache check ---")
print("Number of questions:", len(retrieval_cache))

required_question_fields = [
    "question_id",
    "question",
    "gold_answer",
    "retrieved_passages"
]

cache_errors = []

for i, item in enumerate(retrieval_cache):
    for field in required_question_fields:
        if field not in item:
            cache_errors.append(
                f"Record {i}: missing field '{field}'"
            )

    passages = item.get("retrieved_passages", [])

    if len(passages) != 8:
        cache_errors.append(
            f"Record {i}: expected 8 passages, got {len(passages)}"
        )

    for j, passage in enumerate(passages):
        for field in ["rank", "title", "passage"]:
            if field not in passage:
                cache_errors.append(
                    f"Record {i}, passage {j}: missing '{field}'"
                )

question_ids = [
    item.get("question_id") for item in retrieval_cache
]

if len(question_ids) != len(set(question_ids)):
    cache_errors.append("Duplicate question IDs found")

print("Cache validation errors:", len(cache_errors))

for error in cache_errors[:20]:
    print("-", error)

# 3. Inspect pilot questions
print("\n--- Pilot questions check ---")
questions_df = pd.read_csv(questions_path)

print("Rows:", len(questions_df))
print("Columns:", questions_df.columns.tolist())

# 4. Inspect clean baseline runner output
print("\n--- Clean baseline output check ---")
runner_df = pd.read_csv(runner_path)

print("Rows:", len(runner_df))
print("Columns:", runner_df.columns.tolist())
print("\nFirst 3 rows:")
display(runner_df.head(3))

print("\nMissing values by column:")
print(runner_df.isna().sum())

# 5. Basic consistency checks
print("\n--- Consistency check ---")
print("Cache questions:", len(retrieval_cache))
print("Pilot question rows:", len(questions_df))
print("Runner output rows:", len(runner_df))

if len(retrieval_cache) == 20 and len(questions_df) == 20:
    print("PASS: pilot input sets contain 20 records.")
else:
    print("CHECK: input record counts differ from the expected 20.")

if len(runner_df) == 20:
    print("PASS: runner has 20 output rows.")
else:
    print("CHECK: runner output count differs from 20.")

results/pilot/retrieval_cache.json: FOUND
results/pilot/pilot_questions.csv: FOUND
results/pilot/clean_pilot_runner_v1.csv: FOUND

--- Retrieval cache check ---
Number of questions: 20
Cache validation errors: 0

--- Pilot questions check ---
Rows: 20
Columns: ['question_id', 'question', 'gold_answer']

--- Clean baseline output check ---
Rows: 20
Columns: ['question_id', 'condition', 'method', 'gold_answer', 'predicted_answer', 'EM', 'Token_F1', 'num_retrieved_passages', 'generation_latency_seconds', 'status', 'error']

First 3 rows:


,question_id,condition,method,gold_answer,predicted_answer,EM,Token_F1,num_retrieved_passages,generation_latency_seconds,status,error
0,5a8b57f25542995d1e6f1371,clean,baseline,yes,Yes.,1,1.000000,8,4.035620,success,NaN
1,5a8c7595554299585d9e36b6,clean,baseline,Chief of Protocol,Secretary of State for Constitutional Affairs.,0,0.222222,8,5.103004,success,NaN
2,5a85ea095542994775f606a8,clean,baseline,Animorphs,Animorphs.,1,1.000000,8,8.088143,success,NaN



Missing values by column:
question_id                    0
condition                      0
method                         0
gold_answer                    0
predicted_answer               0
EM                             0
Token_F1                       0
num_retrieved_passages         0
generation_latency_seconds     0
status                         0
error                         20
dtype: int64

--- Consistency check ---
Cache questions: 20
Pilot question rows: 20
Runner output rows: 20
PASS: pilot input sets contain 20 records.
PASS: runner has 20 output rows.


## Step 14（Need adjust)

In [32]:
import pandas as pd


def detector_metrics(true_labels, predicted_labels):
    if len(true_labels) != len(predicted_labels):
        raise ValueError(
            "Labels and predictions must have equal lengths."
        )

    if len(true_labels) == 0:
        raise ValueError("Labels cannot be empty.")

    if not all(label in (0, 1) for label in true_labels):
        raise ValueError("Ground-truth labels must be 0 or 1.")

    if not all(label in (0, 1) for label in predicted_labels):
        raise ValueError("Predictions must be 0 or 1.")

    tp = sum(
        y == 1 and p == 1
        for y, p in zip(true_labels, predicted_labels)
    )

    fp = sum(
        y == 0 and p == 1
        for y, p in zip(true_labels, predicted_labels)
    )

    fn = sum(
        y == 1 and p == 0
        for y, p in zip(true_labels, predicted_labels)
    )

    tn = sum(
        y == 0 and p == 0
        for y, p in zip(true_labels, predicted_labels)
    )

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn
    }


def scan_detector_thresholds(scores, labels, thresholds):
    """
    scores: semantic attack scores
    labels: ground-truth labels (1 = malicious, 0 = benign)
    thresholds: candidate thresholds

    A prediction is malicious when score >= threshold.
    """

    if len(scores) != len(labels):
        raise ValueError(
            "Scores and labels must have equal lengths."
        )

    if len(scores) == 0:
        raise ValueError("Development data cannot be empty.")

    if not thresholds:
        raise ValueError("Thresholds cannot be empty.")

    rows = []

    for threshold in thresholds:
        predictions = [
            int(score >= threshold)
            for score in scores
        ]

        metrics = detector_metrics(labels, predictions)

        rows.append({
            "threshold": threshold,
            **metrics
        })

    return pd.DataFrame(rows)


# Example usage once Vivek provides scores and labels:
# threshold_results = scan_detector_thresholds(
#     scores=development_scores,
#     labels=development_labels,
#     thresholds=[0.30, 0.35, 0.40, 0.45, 0.50]
# )
# display(threshold_results)

In [33]:
from pathlib import Path
import pandas as pd

# Candidate thresholds: an initial grid, not final settings
candidate_thresholds = [
    0.30, 0.35, 0.40, 0.45, 0.50,
    0.55, 0.60, 0.65, 0.70
]

def run_threshold_tuning(scores, labels, thresholds):
    if len(scores) != len(labels):
        raise ValueError("Scores and labels must have equal lengths.")

    if len(scores) == 0:
        raise ValueError("Development data cannot be empty.")

    if any(pd.isna(score) for score in scores):
        raise ValueError("Scores contain missing values.")

    if any(score < 0 or score > 1 for score in scores):
        raise ValueError("Expected semantic scores between 0 and 1.")

    results = scan_detector_thresholds(
        scores=scores,
        labels=labels,
        thresholds=thresholds
    )

    results = results.sort_values(
        by=["f1", "precision"],
        ascending=[False, False]
    ).reset_index(drop=True)

    output_path = Path("results/development")
    output_path.mkdir(parents=True, exist_ok=True)

    results.to_csv(
        output_path / "threshold_scan_v1.csv",
        index=False
    )

    return results

# Run only after Vivek supplies the development scores and labels:
#
# threshold_results = run_threshold_tuning(
#     scores=development_scores,
#     labels=development_labels,
#     thresholds=candidate_thresholds
# )
#
# display(threshold_results)

In [34]:
import pandas as pd
from pathlib import Path

pilot_path = Path("results/pilot/pilot_questions.csv")

pilot_df = pd.read_csv(pilot_path)

print("Pilot columns:", pilot_df.columns.tolist())
display(pilot_df.head(3))

print("Validation dataset columns:", dataset["validation"].column_names)
print("Validation dataset size:", len(dataset["validation"]))

Pilot columns: ['question_id', 'question', 'gold_answer']


,question_id,question,gold_answer
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same ...,yes
1,5a8c7595554299585d9e36b6,What government position was held by the woman...,Chief of Protocol
2,5a85ea095542994775f606a8,"What science fantasy young adult series, told ...",Animorphs


NameError: name 'dataset' is not defined

In [ ]:
import random
import pandas as pd
from pathlib import Path

# Reproducible split
SEED = 764
N_DEVELOPMENT = 40
N_FINAL_TEST = 100

output_dir = Path("results/splits")
output_dir.mkdir(parents=True, exist_ok=True)

# Identify the question ID column in the pilot file
possible_id_columns = ["question_id", "id"]
pilot_id_col = next(
    (col for col in possible_id_columns if col in pilot_df.columns),
    None
)

if pilot_id_col is None:
    raise ValueError(
        "Could not identify the question ID column in pilot_questions.csv. "
        "Check the printed column names before continuing."
    )

pilot_ids = set(pilot_df[pilot_id_col].astype(str))

# Convert the validation dataset to a DataFrame
validation_df = dataset["validation"].to_pandas()

required_columns = {"id", "question", "answer"}
missing_columns = required_columns - set(validation_df.columns)

if missing_columns:
    raise ValueError(
        f"Validation dataset is missing columns: {missing_columns}"
    )

validation_df["id"] = validation_df["id"].astype(str)

# Exclude all existing pilot questions
eligible_df = validation_df[
    ~validation_df["id"].isin(pilot_ids)
].copy()

if len(eligible_df) < N_DEVELOPMENT + N_FINAL_TEST:
    raise ValueError("Not enough eligible questions for the requested split.")

# Deterministic shuffle, then create disjoint sets
shuffled_df = eligible_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

development_df = shuffled_df.iloc[:N_DEVELOPMENT].copy()
final_test_df = shuffled_df.iloc[
    N_DEVELOPMENT:N_DEVELOPMENT + N_FINAL_TEST
].copy()

# Keep only the question-level fields needed for the split
columns_to_save = ["id", "question", "answer"]

development_df[columns_to_save].rename(
    columns={"id": "question_id", "answer": "gold_answer"}
).to_csv(
    output_dir / "development_questions.csv",
    index=False
)

final_test_df[columns_to_save].rename(
    columns={"id": "question_id", "answer": "gold_answer"}
).to_csv(
    output_dir / "final_test_questions.csv",
    index=False
)

# Verify the split
dev_ids = set(development_df["id"])
test_ids = set(final_test_df["id"])

assert len(development_df) == N_DEVELOPMENT
assert len(final_test_df) == N_FINAL_TEST
assert dev_ids.isdisjoint(test_ids)
assert dev_ids.isdisjoint(pilot_ids)
assert test_ids.isdisjoint(pilot_ids)

print("Development questions:", len(development_df))
print("Reserved final-test questions:", len(final_test_df))
print("Overlap between development and test:", len(dev_ids & test_ids))
print("Overlap with pilot questions:", len((dev_ids | test_ids) & pilot_ids))
print("Saved to:", output_dir)

## Step 16（need adjust)

In [35]:
from pathlib import Path
import json

# Step 16: Final experiment plan
# This cell defines the experiment design only.
# It does NOT execute experiments or generate results.

FINAL_QUESTION_TARGET = 100
MINIMUM_QUESTION_TARGET = 80
TOP_K = 8

METHODS = {
    "M0": {
        "name": "No defence",
        "detector": None,
        "granularity": "none",
        "status": "runner_scaffold_only",
    },
    "M1": {
        "name": "Rule detector + sentence removal",
        "detector": "rule",
        "granularity": "sentence",
        "status": "BLOCKED: detector integration pending",
    },
    "M2": {
        "name": "Embedding detector + sentence removal",
        "detector": "embedding",
        "granularity": "sentence",
        "status": "BLOCKED: detector integration pending",
    },
    "M3": {
        "name": "Combined detector + whole-passage removal",
        "detector": "combined",
        "granularity": "passage",
        "status": "BLOCKED: combined detector integration pending",
    },
    "M4": {
        "name": "Combined detector + sentence removal",
        "detector": "combined",
        "granularity": "sentence",
        "status": "BLOCKED: combined detector integration pending",
    },
}

CONDITIONS = [
    "clean",
    "explicit_attack",
    "heldout_paraphrased_topical_attack",
]

experiment_plan = {
    "step": 16,
    "name": "Final Experiments",
    "preferred_question_count": FINAL_QUESTION_TARGET,
    "minimum_question_count": MINIMUM_QUESTION_TARGET,
    "top_k": TOP_K,
    "methods": METHODS,
    "conditions": CONDITIONS,
    "final_test_ids_frozen": False,
    "thresholds_frozen": False,
    "attack_data_ready": False,
    "detector_modules_integrated": False,
    "formal_execution_authorized": False,
}

output_dir = Path("results/final_experiments")
output_dir.mkdir(parents=True, exist_ok=True)

plan_path = output_dir / "experiment_plan_v1.json"

with open(plan_path, "w", encoding="utf-8") as f:
    json.dump(experiment_plan, f, indent=2, ensure_ascii=False)

print("Experiment plan saved to:", plan_path)
print("Planned method-condition combinations:",
      len(METHODS) * len(CONDITIONS))
print("Formal execution: NOT STARTED")

Experiment plan saved to: results/final_experiments/experiment_plan_v1.json
Planned method-condition combinations: 15
Formal execution: NOT STARTED


In [36]:
import pandas as pd
from pathlib import Path

FINAL_RESULT_COLUMNS = [
    "question_id",
    "condition",
    "method",
    "question",
    "gold_answer",
    "model_answer",
    "attack_id",
    "attack_type",
    "retrieved_passage_count",
    "removed_passage_count",
    "removed_sentence_count",
    "supporting_evidence_before",
    "supporting_evidence_retained",
    "attack_success",
    "exact_match",
    "token_f1",
    "false_removal_count",
    "sanitisation_latency_ms",
    "generation_latency_ms",
    "error",
]

results_dir = Path("results/final_experiments")
results_dir.mkdir(parents=True, exist_ok=True)

# Create an empty schema template only.
# Do not treat this file as experimental evidence.
template_path = results_dir / "final_results_template.csv"

pd.DataFrame(columns=FINAL_RESULT_COLUMNS).to_csv(
    template_path, index=False
)

print("Result schema saved to:", template_path)
print("Rows:", 0)
print("This is an empty template, not experimental results.")

Result schema saved to: results/final_experiments/final_results_template.csv
Rows: 0
This is an empty template, not experimental results.


In [37]:
from pathlib import Path
import json
import pandas as pd

# Step 16: Check readiness for formal experiments.
# This cell checks files only. It does NOT run the experiments.

required_existing_files = {
    "Experiment plan": Path(
        "results/final_experiments/experiment_plan_v1.json"
    ),
    "Result template": Path(
        "results/final_experiments/final_results_template.csv"
    ),
    "Pilot questions": Path(
        "results/pilot/pilot_questions.csv"
    ),
    "Retrieval cache": Path(
        "results/pilot/retrieval_cache.json"
    ),
    "Clean pilot runner output": Path(
        "results/pilot/clean_pilot_runner_v1.csv"
    ),
}

print("=== Existing file checks ===")
for name, path in required_existing_files.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"[{status}] {name}: {path}")

print("\n=== Experiment configuration ===")
plan_path = required_existing_files["Experiment plan"]

if plan_path.exists():
    with open(plan_path, "r", encoding="utf-8") as f:
        plan = json.load(f)

    print("Methods:", ", ".join(plan["methods"].keys()))
    print("Conditions:", ", ".join(plan["conditions"]))
    print("Preferred question count:",
          plan["preferred_question_count"])
    print("Minimum fallback:",
          plan["minimum_question_count"])

print("\n=== Formal execution blockers ===")

blockers = [
    "Final-test question IDs are not yet frozen.",
    "Development-tuned detector thresholds are not confirmed frozen.",
    "Attack datasets and injection interfaces are not integrated.",
    "M1-M4 detector/defence modules are not integrated.",
    "End-to-end pilot integration test has not been completed.",
]

for item in blockers:
    print("[BLOCKED]", item)

print("\nFormal experiments started: NO")
print("Existing pilot results will not be modified.")

=== Existing file checks ===
[FOUND] Experiment plan: results/final_experiments/experiment_plan_v1.json
[FOUND] Result template: results/final_experiments/final_results_template.csv
[FOUND] Pilot questions: results/pilot/pilot_questions.csv
[FOUND] Retrieval cache: results/pilot/retrieval_cache.json
[FOUND] Clean pilot runner output: results/pilot/clean_pilot_runner_v1.csv

=== Experiment configuration ===
Methods: M0, M1, M2, M3, M4
Conditions: clean, explicit_attack, heldout_paraphrased_topical_attack
Preferred question count: 100
Minimum fallback: 80

=== Formal execution blockers ===
[BLOCKED] Final-test question IDs are not yet frozen.
[BLOCKED] Development-tuned detector thresholds are not confirmed frozen.
[BLOCKED] Attack datasets and injection interfaces are not integrated.
[BLOCKED] M1-M4 detector/defence modules are not integrated.
[BLOCKED] End-to-end pilot integration test has not been completed.

Formal experiments started: NO
Existing pilot results will not be modified.


## step17 (need adjust)

In [38]:
import re
from collections import Counter

def normalize_answer(text):
    """Normalize an answer for Exact Match and Token F1."""
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = " ".join(text.split())
    return text


def exact_match(prediction, gold_answer):
    """Return 1 if normalized answers match, otherwise 0."""
    return int(
        normalize_answer(prediction)
        == normalize_answer(gold_answer)
    )


def token_f1(prediction, gold_answer):
    """Calculate token-level F1 using token overlap."""
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold_answer).split()

    if not pred_tokens or not gold_tokens:
        return int(pred_tokens == gold_tokens)

    overlap = Counter(pred_tokens) & Counter(gold_tokens)
    common = sum(overlap.values())

    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


def evaluate_answer(prediction, gold_answer):
    """Calculate answer-quality metrics for one prediction."""
    return {
        "exact_match": exact_match(prediction, gold_answer),
        "token_f1": token_f1(prediction, gold_answer),
    }

In [39]:
# test (artifical examples)
test_cases = [
    ("yes", "yes"),
    ("The answer is Paris.", "Paris"),
    ("New York", "New York City"),
    ("apple", "banana"),
]

for prediction, gold in test_cases:
    print({
        "prediction": prediction,
        "gold_answer": gold,
        **evaluate_answer(prediction, gold),
    })

{'prediction': 'yes', 'gold_answer': 'yes', 'exact_match': 1, 'token_f1': 1.0}
{'prediction': 'The answer is Paris.', 'gold_answer': 'Paris', 'exact_match': 0, 'token_f1': 0.5}
{'prediction': 'New York', 'gold_answer': 'New York City', 'exact_match': 0, 'token_f1': 0.8}
{'prediction': 'apple', 'gold_answer': 'banana', 'exact_match': 0, 'token_f1': 0.0}


In [40]:
# binary classification
def evaluate_binary_classification(true_labels, predicted_labels):
    """Calculate precision, recall and F1 for binary labels."""
    if len(true_labels) != len(predicted_labels):
        raise ValueError("Label lengths do not match.")

    if not true_labels:
        raise ValueError("Labels cannot be empty.")

    if any(x not in (0, 1) for x in true_labels):
        raise ValueError("True labels must be 0 or 1.")

    if any(x not in (0, 1) for x in predicted_labels):
        raise ValueError("Predicted labels must be 0 or 1.")

    tp = sum(
        y == 1 and p == 1
        for y, p in zip(true_labels, predicted_labels)
    )
    fp = sum(
        y == 0 and p == 1
        for y, p in zip(true_labels, predicted_labels)
    )
    fn = sum(
        y == 1 and p == 0
        for y, p in zip(true_labels, predicted_labels)
    )
    tn = sum(
        y == 0 and p == 0
        for y, p in zip(true_labels, predicted_labels)
    )

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }

In [41]:
# test
# Synthetic sanity check only — not project results.
test_true = [1, 1, 0, 0]
test_pred = [1, 0, 0, 1]

print(evaluate_binary_classification(test_true, test_pred))

{'precision': 0.5, 'recall': 0.5, 'f1': 0.5, 'tp': 1, 'fp': 1, 'fn': 1, 'tn': 1}


## step 18 (need adjust)

In [42]:
from pathlib import Path
import pandas as pd

ERROR_ANALYSIS_COLUMNS = [
    "question_id",
    "condition",
    "method",
    "gold_answer",
    "model_answer",
    "exact_match",
    "token_f1",
    "attack_success",
    "retrieval_failure",
    "detector_false_negative",
    "detector_false_positive",
    "evidence_removed",
    "execution_error",
    "primary_error_category",
    "error_explanation",
    "supporting_evidence_notes",
    "comparison_notes",
]

analysis_dir = Path("results/error_analysis")
analysis_dir.mkdir(parents=True, exist_ok=True)

template_path = analysis_dir / "error_analysis_template.csv"

pd.DataFrame(columns=ERROR_ANALYSIS_COLUMNS).to_csv(
    template_path,
    index=False
)

print("Error analysis template created:", template_path)
print("Number of columns:", len(ERROR_ANALYSIS_COLUMNS))
print("Rows:", 0)
print("This is a template, not experimental evidence.")

Error analysis template created: results/error_analysis/error_analysis_template.csv
Number of columns: 17
Rows: 0
This is a template, not experimental evidence.


In [43]:
def summarize_error_categories(error_df):
    """
    Summarize error-category counts by method and condition.

    Expected input:
        One row per evaluated question-method-condition record.
        Error category columns should contain 0/1 or boolean values.
    """
    required_columns = [
        "method",
        "condition",
        "answer_error",
        "retrieval_failure",
        "attack_success",
        "detector_false_negative",
        "detector_false_positive",
        "evidence_removed",
        "execution_error",
    ]

    missing = [
        col for col in required_columns
        if col not in error_df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    error_columns = [
        "answer_error",
        "retrieval_failure",
        "attack_success",
        "detector_false_negative",
        "detector_false_positive",
        "evidence_removed",
        "execution_error",
    ]

    return (
        error_df
        .groupby(["method", "condition"], dropna=False)[error_columns]
        .sum()
        .reset_index()
    )

In [44]:
from pathlib import Path
import pandas as pd

pilot_output_path = Path(
    "results/pilot/clean_pilot_runner_v1.csv"
)

if not pilot_output_path.exists():
    raise FileNotFoundError(
        f"Pilot output not found: {pilot_output_path}"
    )

pilot_output = pd.read_csv(pilot_output_path)

print("=== Pilot output overview ===")
print("Shape:", pilot_output.shape)
print("Columns:")
for column in pilot_output.columns:
    print(" -", column)

print("\n=== First 3 records ===")
display(pilot_output.head(3))

print("\n=== Missing-value counts ===")
display(pilot_output.isna().sum().to_frame("missing_count"))

=== Pilot output overview ===
Shape: (20, 11)
Columns:
 - question_id
 - condition
 - method
 - gold_answer
 - predicted_answer
 - EM
 - Token_F1
 - num_retrieved_passages
 - generation_latency_seconds
 - status
 - error

=== First 3 records ===


,question_id,condition,method,gold_answer,predicted_answer,EM,Token_F1,num_retrieved_passages,generation_latency_seconds,status,error
0,5a8b57f25542995d1e6f1371,clean,baseline,yes,Yes.,1,1.000000,8,4.035620,success,NaN
1,5a8c7595554299585d9e36b6,clean,baseline,Chief of Protocol,Secretary of State for Constitutional Affairs.,0,0.222222,8,5.103004,success,NaN
2,5a85ea095542994775f606a8,clean,baseline,Animorphs,Animorphs.,1,1.000000,8,8.088143,success,NaN



=== Missing-value counts ===


,missing_count
question_id,0
condition,0
method,0
gold_answer,0
predicted_answer,0
EM,0
Token_F1,0
num_retrieved_passages,0
generation_latency_seconds,0
status,0
